# `QUAX`: Accelerated Quantum Information

This notebook demonstrates the `quax` module which allows for hardware-accelerated and differentiable calculations using quantum states, gates and superoperators.

### Quantum Objects: States, operators and superoperators

Quantum objects are sorted into three categories: `State`, `Operator` and `Superoperator`


`State` includes `StateVector` and `DensityMatrix`

`Operator` includes `Unitary` and `Kraus`

`Superoperator` includes `SuperOp`, `Choi`, `PauliLiouville`, `Chi`

### Operator syntax

For all objects, the following operations are defined:

- `@ `   : composition / action
- `| `   : tensor product
- `* `   : scalar multiplication
- `**`  : powers
- `- `   : negation

Not all operations are defined for all combinations of objects. For example, while we can tensor product any two `Superoperator` objects, we can't tensor product a `Choi` and `DensityMatrix` - this doesn't make sense.

When we operator with two compatible objects of different types, for example `Choi @ SuperOp`, the result will generally be the type of the right object - so in this case a `Choi`. Only the forward `__matmul__` is defined.

#### Examples

Apply a `Unitary` to a `StateVector`

```python
initial_psi = zero_state_vector(2)
psi = CZ @ initial_psi
```

Apply a sequence of `Unitary` to a `DensityMatrix`

```python
mixed_rho = mixed_state_matrix(2)
rho = CZ @ (RX(jnp.pi/2) | RZ(jnp.pi / 2)) @ mixed_rho
```

Apply a `Unitary` to an `DensityMatrix` ensemble. Note here that the ensemble indices are leading, the final two dimensions should always be the density matrix. The dims remain the same - one requirement of this kind of batched operations is that the density matrices are of the same size.

```python
mixed_rhos = DensityMatrix(
    data=jnp.array([mixed_state_matrix(2).data, mixed_state_matrix(2).data]),
    dims=(2, 2),
)
rhos = CZ @ mixed_rhos
```

### Just-in-time compilation (JIT)

JIT is an essential part of Jax. Not only is it essential for achieving high performance, but it's a hard requirement for using jax's unique capabilities like `vmap` and `grad`.

JIT comes with strings attached - specifically, the dimensions of the calculation must be defined beforehand. Operators, states and superoperators have various dimensions depending both on the number of qubits and their dimension (qutrits, quarts etc.). The dimensions are always carried by the states and operators, and JITs will automatically be specific to those dimensions which are static args for the functions. Changing the dimensions of operator or state will result in a new JIT.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
import jax.numpy as jnp

import quax as qx
from quax.gates import CZ, ISWAP, RX, RZ

## States

Let's start by initializing some states.

In [ ]:
initial_psi = qx.zero_state_vector(3)
initial_rho = qx.zero_state_matrix(3)
mixed_rho = qx.mixed_state_matrix(2)

In [ ]:
mixed_rho

## Operators

We can also use the standard pyquil gates

In [ ]:
RX(jnp.pi / 2)

Unitaries can be composed

In [ ]:
CZ @ ISWAP

They can also be tensored

In [ ]:
RX(jnp.pi / 2) | RZ(jnp.pi / 4)

They can be multiplied. Note that multiplying a Unitary by a complex number not equal to 1 will result in a Kraus

In [ ]:
1.0 * CZ

In [ ]:
0.5 * CZ

They can be fractionally powered

In [ ]:
CZ ** (0.5)

Operations can also be broadcast

### We can act on states using unitary matrices

Naturally, we can also apply a unitary matrix to a state

In [ ]:
initial_psi = qx.zero_state_vector(2)
qx.random_state_vector(dims=(2,), key=jax.random.PRNGKey(0))
CZ @ initial_psi

We can also act on mixed states.

In [ ]:
initial_rho = qx.mixed_state_matrix(2)

CZ @ initial_rho

Or even an ensemble of states

In [ ]:
size = (3, 5)

rhos = qx.random_density_matrix(rank=2, dims=(2, 2), key=jax.random.PRNGKey(0), size=size)

rho_outs = jax.vmap(lambda rho: CZ @ rho, in_axes=0, out_axes=0)(rhos)
print(rho_outs)

## Superoperators

Superoperators describe noisy operations. We can promote a unitary to a superoperator.

In [ ]:
choi = qx.random_choi_BCSZ(dims=((2, 2, 2), (2, 2, 2)), rank=4, key=jax.random.key(1))

In [ ]:
S = qx.unitary_to_superop(CZ)
print(S)

### Common channels

We can construct some common superoperators such as bit flips, amplitude damping and depolarizing.

In [ ]:
S = qx.depolarizing_channel_superoperator(0.1, 1)
print(S)

### Conversions

In [ ]:
C = qx.superop_to_choi(S)
print(C)

P = qx.superop_to_pauli_liouville(S)
print(P)

### Composing channels

We can compose superoperators using the `@` symbol

In [ ]:
print(S @ S)

We can also compose with Superoperators in different forms. By convention, the output type will be that of the lefthand operator.

Ie,

`SuperOp @ Choi -> SuperOp`

`Choi @ SuperOp -> Choi`

In [ ]:
S @ C

In [ ]:
C @ S

In [ ]:
P @ S @ C

### Tensoring Channels

Similarly, we can tensor channels

In [ ]:
PxP = P | P
print(PxP)

In [ ]:
PxCxS = P | C | S
print(PxCxS)

## Distance Metrics

### State Fidelity

In [ ]:
rho = qx.zero_state_matrix(2)
sigma = qx.mixed_state_matrix(2)

print(f"fidelity(𝜌, 𝜌) = {qx.fidelity(rho, rho):.2f}")
print(f"fidelity(𝜌, 𝜎) = {qx.fidelity(rho, sigma):.2f}")
print(f"fidelity(𝜎, 𝜎) = {qx.fidelity(sigma, sigma):.2f}")

In [ ]:
choi = qx.unitary_to_choi(RX(jnp.pi))

print(f"Process fidelity (self): {qx.process_fidelity(choi, choi):.2f}")

print(f"Process fidelity (identity): {qx.process_fidelity(choi):.2f}")

## Random Operators and Ensembles

Random operators have various uses in quantum information. We provide common ensembles and distributions of operators.

`random_unitary`: Sample random unitaries from the Haar measure.

In [ ]:
key = jax.random.key(4258)
num_unitaries = (80, 22, 64)
dims = ((2,), (2,))
unitaries = qx.random_unitary(dims=dims, key=key, size=num_unitaries)

In [ ]:
qx.is_two_design(unitaries, atol=1e-2)

We can check properties of the ensemble: For example it's 1-design and 2-design properties

In [ ]:
qx.is_two_design(qx.ensembles.CLIFFORD_ENSEMBLE, atol=1e-2)

In [ ]:
qx.is_two_design(qx.ensembles.TETRAHEDRAL_ENSEMBLE, atol=1e-2)